
1. LEITURA DA CAMADA RAW

Nesta etapa, o arquivo CSV é lido diretamente do Volume Raw.
Os dados ainda não sofrem transformações de qualidade.

In [0]:
# Leitura do arquivo CSV armazenado no Volume Raw.
# O cabeçalho é utilizado como nome das colunas e o Spark infere os tipos de dados.
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("sep", ",") \
    .csv("/Volumes/salary_mvp/salary/raw/salary.csv")


2. DIAGNÓSTICO INICIAL DOS DADOS

Nesta etapa, são realizadas verificações exploratórias
para compreender a estrutura e a qualidade dos dados
antes da criação da camada Bronze.

In [0]:
# Visualização inicial dos dados após a ingestão,
# permitindo verificar a estrutura e o conteúdo do arquivo.
display(df)

In [0]:
# Verificação da quantidade de registros e colunas
# carregados a partir do arquivo original.
print("Linhas:", df.count())
print("Colunas:", len(df.columns))

In [0]:
# Listagem das colunas presentes no conjunto de dados.
print(df.columns)

In [0]:
# Verificação dos tipos de dados inferidos pelo Spark para cada coluna.
df.printSchema()

In [0]:
# Verificação da quantidade de valores nulos em cada coluna.
# Essa análise faz parte do diagnóstico inicial da qualidade dos dados.
from pyspark.sql.functions import col, sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

In [0]:
# Identificação de valores ausentes representados pelo caractere "?".
# Diferentemente de NULL, esses valores estão armazenados como texto.
from pyspark.sql.functions import trim

for c in df.columns:
    qtd = df.filter(trim(col(c)) == "?").count()
    if qtd > 0:
        print(c, "->", qtd)

In [0]:
# Verificação de registros duplicados considerando todas as colunas.
# A comparação entre o total original e o total após dropDuplicates()
# permite identificar a quantidade de registros repetidos.
print("Total de linhas:", df.count())
print("Linhas após remoção de duplicadas:", df.dropDuplicates().count())
print("Linhas duplicadas:", df.count() - df.dropDuplicates().count())

In [0]:
# Verificação da distribuição da variável salary,
# que será utilizada como referência nas análises de negócio.
display(
    df.groupBy("salary")
      .count()
      .orderBy("salary")
)

3. CAMADA BRONZE

Nesta etapa, os dados são persistidos em formato Delta,
preservando os dados originalmente ingeridos.
Não são aplicadas regras de limpeza ou transformação.

In [0]:
# Persistência dos dados brutos na camada Bronze.
# Nesta etapa, os dados são armazenados sem aplicação de regras de limpeza ou transformação.

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("salary_mvp.salary.bronze_salary")

4. VALIDAÇÃO DA CAMADA BRONZE

Após a persistência, a tabela Bronze é consultada para
confirmar que os dados foram armazenados corretamente.

In [0]:
# Consulta da tabela Bronze para validar a persistência dos dados.
display(
    spark.table("salary_mvp.salary.bronze_salary")
)

In [0]:
# Validação da quantidade de registros armazenados na camada Bronze.
bronze_df = spark.table("salary_mvp.salary.bronze_salary")

print("Linhas na Bronze:", bronze_df.count())
print("Colunas na Bronze:", len(bronze_df.columns))

5. CAMADA SILVER

A camada Silver é construída a partir dos dados persistidos na Bronze.
Nesta etapa são aplicadas regras de tratamento e qualidade
dos dados, sem alterar a camada Bronze.

In [0]:
# Leitura dos dados armazenados na camada Bronze.
# A Silver será construída a partir dos dados persistidos nessa camada.

bronze_df = spark.table("salary_mvp.salary.bronze_salary")

In [0]:
# Criação da camada Silver a partir da Bronze.
# Nesta etapa, os valores ausentes representados por "?"
# são convertidos para NULL e os textos são padronizados.

from pyspark.sql.functions import col, trim, when

silver_df = bronze_df

# Substitui "?" por NULL nas colunas que possuem valores ausentes.
for c in ["workclass", "occupation", "native-country"]:
    silver_df = silver_df.withColumn(
        c,
        when(trim(col(c)) == "?", None).otherwise(trim(col(c)))
    )

In [0]:
# Remoção de registros completamente duplicados.
# A comparação considera todas as colunas do conjunto de dados.

silver_df = silver_df.dropDuplicates()

In [0]:
# Validação da quantidade de registros após a remoção das duplicidades.

print("Linhas na Silver:", silver_df.count())
print("Colunas na Silver:", len(silver_df.columns))

In [0]:
# Verificação da existência de registros duplicados após o tratamento.

duplicados_silver = silver_df.count() - silver_df.dropDuplicates().count()

print("Linhas duplicadas na Silver:", duplicados_silver)

6.1 Persistência da Silver

In [0]:
# Persistência dos dados tratados na camada Silver.
# Os dados são armazenados em formato Delta para as etapas seguintes do pipeline.

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("salary_mvp.salary.silver_salary")

6.2 Validação da Silver

In [0]:
# Leitura da tabela Silver para validar a persistência dos dados.

silver_df = spark.table("salary_mvp.salary.silver_salary")

display(silver_df)

In [0]:
from pyspark.sql.functions import col, sum, when

silver_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silver_df.columns
]).show()

In [0]:
# Validação final da quantidade de registros e colunas da tabela Silver.

print("Linhas na Silver:", silver_df.count())
print("Colunas na Silver:", len(silver_df.columns))

In [0]:
# Leitura dos dados tratados da camada Silver.
# A camada Gold será construída a partir dos dados já validados.

silver_df = spark.table("salary_mvp.salary.silver_salary")

In [0]:
# Verificação da distribuição da variável salary na camada Silver.

display(
    silver_df.groupBy("salary")
             .count()
             .orderBy("salary")
)

 7.1 Transformações da Gold

A camada Gold é construída a partir dos dados tratados na Silver e recebe transformações voltadas ao consumo analítico.

Nesta etapa, a variável salary é padronizada e é criada a variável salary_binary, que representa a faixa de renda
de forma numérica:

0 = até 50K

1 = acima de 50K.

In [0]:
gold_df = silver_df

In [0]:
# Criação da camada Gold.
# A coluna salary é padronizada e convertida em uma representação numérica.
# Essa transformação facilita análises posteriores e possíveis modelos.

from pyspark.sql.functions import col, trim, when

gold_df = silver_df.withColumn(
    "salary",
    trim(col("salary"))
).withColumn(
    "salary_binary",
    when(col("salary") == "<=50K", 0)
    .when(col("salary") == ">50K", 1)
)

In [0]:
# Validação da transformação da variável salary.

display(
    gold_df.groupBy("salary", "salary_binary")
           .count()
           .orderBy("salary")
)

Criação de uma representação textual da faixa de renda, facilitando a interpretação dos resultados nas análises.

A variável salary_range mantém as duas categorias originais em uma descrição mais legível:
Até 50K e Acima de 50K.

In [0]:
# Criação de uma categoria descritiva para facilitar
# a interpretação da faixa de renda nas análises.

gold_df = gold_df.withColumn(
    "salary_range",
    when(col("salary") == "<=50K", "Até 50K")
    .when(col("salary") == ">50K", "Acima de 50K")
)

In [0]:
display(
    gold_df.select(
        "salary",
        "salary_binary",
        "salary_range"
    ).limit(20)
)

In [0]:
display(gold_df)

In [0]:
gold_df.printSchema()

 7.2 Validação das transformações da Gold

Verificação da distribuição das categorias de renda após as transformações realizadas na camada Gold.

In [0]:
display(
    gold_df.groupBy("salary_range")
           .count()
           .orderBy("salary_range")
)


 7.3 Persistência da Gold

Os dados transformados na camada Gold são persistidos em formato Delta para disponibilizar uma tabela estruturada para as análises de negócio.

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("salary_mvp.salary.gold_salary")

 7.4 Validação da Gold

Leitura da tabela Gold persistida para verificar se os dados foram armazenados corretamente e se a quantidade de registros permanece consistente com a camada Silver.

In [0]:
gold_df = spark.table("salary_mvp.salary.gold_salary")

print("Linhas na Gold:", gold_df.count())
print("Colunas na Gold:", len(gold_df.columns))

In [0]:
display(gold_df)

 8. ANÁLISE DE DADOS

Nesta etapa, a camada Gold é utilizada para responder às perguntas de negócio definidas no projeto.

As análises são realizadas a partir dos dados tratados e persistidos na camada Gold.

 8.1 Pergunta 1 — Escolaridade

Como a faixa de renda varia de acordo com o nível de escolaridade dos indivíduos?

In [0]:
display(
    gold_df.groupBy("education", "salary_range")
           .count()
           .orderBy("education", "salary_range")
)

In [0]:
# Percentual de indivíduos com renda acima de 50K
# dentro de cada nível de escolaridade.

from pyspark.sql.functions import count, sum, round

display(
    gold_df.groupBy("education")
           .agg(
               count("*").alias("total"),
               sum("salary_binary").alias("acima_50k"),
               round(
                   (sum("salary_binary") / count("*")) * 100,
                   2
               ).alias("percentual_acima_50k")
           )
           .orderBy("percentual_acima_50k", ascending=False)
)

In [0]:
# Gráfico da proporção de indivíduos com renda acima de 50K
# por nível de escolaridade.

display(
    gold_df.groupBy("education")
           .agg(
               round(
                   (sum("salary_binary") / count("*")) * 100,
                   2
               ).alias("percentual_acima_50k")
           )
           .orderBy("percentual_acima_50k", ascending=False)
)

Databricks visualization. Run in Databricks to view.

 8.2 Pergunta 2 — Ocupação

Quais ocupações apresentam as maiores proporções de indivíduos com renda anual superior a 50K?

In [0]:
# Percentual de indivíduos com renda acima de 50K
# dentro de cada ocupação.
#
# Os valores NULL representam registros cuja ocupação
# não estava informada na base original.
# Esses registros são mantidos na tabela para fins de
# transparência, mas não são considerados uma ocupação
# na interpretação dos resultados.

from pyspark.sql.functions import count, sum, round

display(
    gold_df.groupBy("occupation")
           .agg(
               count("*").alias("total"),
               sum("salary_binary").alias("acima_50k"),
               round(
                   (sum("salary_binary") / count("*")) * 100,
                   2
               ).alias("percentual_acima_50k")
           )
           .orderBy("percentual_acima_50k", ascending=False)
)

In [0]:
# Dados utilizados para a visualização.
# Registros sem ocupação informada (NULL) são excluídos
# do gráfico, pois não representam uma categoria ocupacional.

occupation_chart = (
    gold_df
    .filter(gold_df["occupation"].isNotNull())
    .groupBy("occupation")
    .agg(
        round(
            (sum("salary_binary") / count("*")) * 100,
            2
        ).alias("percentual_acima_50k")
    )
    .orderBy("percentual_acima_50k", ascending=False)
)

display(occupation_chart)

Databricks visualization. Run in Databricks to view.

 8.3 Pergunta 3 — Jornada

A quantidade de horas trabalhadas por semana está associada à proporção de indivíduos com renda anual superior a 50K?

In [0]:
# Percentual de indivíduos com renda acima de 50K
# para cada quantidade de horas trabalhadas por semana.
#
# A análise permite verificar se a proporção de indivíduos
# com renda acima de 50K varia de acordo com a jornada semanal.

from pyspark.sql.functions import count, sum, round

display(
    gold_df.groupBy("`hours-per-week`")
           .agg(
               count("*").alias("total"),
               sum("salary_binary").alias("acima_50k"),
               round(
                   (sum("salary_binary") / count("*")) * 100,
                   2
               ).alias("percentual_acima_50k")
           )
           .orderBy("`hours-per-week`")
)

In [0]:
# Percentual de indivíduos com renda acima de 50K
# dentro de diferentes faixas de horas trabalhadas por semana.
#
# A criação de faixas reduz o impacto de grupos com poucos
# registros e facilita a identificação de uma possível associação
# entre jornada semanal e renda.

from pyspark.sql.functions import when, count, sum, round

hours_analysis = (
    gold_df
    .withColumn(
        "faixa_horas",
        when(gold_df["hours-per-week"] <= 20, "Até 20h")
        .when(gold_df["hours-per-week"] <= 30, "21-30h")
        .when(gold_df["hours-per-week"] <= 40, "31-40h")
        .when(gold_df["hours-per-week"] <= 50, "41-50h")
        .when(gold_df["hours-per-week"] <= 60, "51-60h")
        .when(gold_df["hours-per-week"] <= 70, "61-70h")
        .when(gold_df["hours-per-week"] <= 80, "71-80h")
        .otherwise("Acima de 80h")
    )
    .withColumn(
        "ordem_faixa",
        when(gold_df["hours-per-week"] <= 20, 1)
        .when(gold_df["hours-per-week"] <= 30, 2)
        .when(gold_df["hours-per-week"] <= 40, 3)
        .when(gold_df["hours-per-week"] <= 50, 4)
        .when(gold_df["hours-per-week"] <= 60, 5)
        .when(gold_df["hours-per-week"] <= 70, 6)
        .when(gold_df["hours-per-week"] <= 80, 7)
        .otherwise(8)
    )
    .groupBy("faixa_horas", "ordem_faixa")
    .agg(
        count("*").alias("total"),
        sum("salary_binary").alias("acima_50k"),
        round(
            (sum("salary_binary") / count("*")) * 100,
            2
        ).alias("percentual_acima_50k")
    )
    .orderBy("ordem_faixa")
    .drop("ordem_faixa")
)

display(hours_analysis)

In [0]:
# Percentual de indivíduos com renda acima de 50K
# dentro de diferentes faixas de horas trabalhadas por semana.

from pyspark.sql.functions import when, count, sum, round, col

hours_analysis = (
    gold_df
    .withColumn(
        "faixa_horas",
        when(col("hours-per-week") <= 20, "Até 20h")
        .when(col("hours-per-week") <= 30, "21-30h")
        .when(col("hours-per-week") <= 40, "31-40h")
        .when(col("hours-per-week") <= 50, "41-50h")
        .when(col("hours-per-week") <= 60, "51-60h")
        .when(col("hours-per-week") <= 70, "61-70h")
        .when(col("hours-per-week") <= 80, "71-80h")
        .otherwise("Acima de 80h")
    )
    .withColumn(
        "ordem_faixa",
        when(col("hours-per-week") <= 20, 1)
        .when(col("hours-per-week") <= 30, 2)
        .when(col("hours-per-week") <= 40, 3)
        .when(col("hours-per-week") <= 50, 4)
        .when(col("hours-per-week") <= 60, 5)
        .when(col("hours-per-week") <= 70, 6)
        .when(col("hours-per-week") <= 80, 7)
        .otherwise(8)
    )
    .groupBy("faixa_horas", "ordem_faixa")
    .agg(
        count("*").alias("total"),
        sum("salary_binary").alias("acima_50k"),
        round(
            (sum("salary_binary") / count("*")) * 100,
            2
        ).alias("percentual_acima_50k")
    )
    .orderBy("ordem_faixa")
    .select("faixa_horas", "percentual_acima_50k")
)

display(hours_analysis)

Databricks visualization. Run in Databricks to view.

 8.4 Pergunta 4 — Escolaridade + ocupação

A relação entre escolaridade e faixa de renda varia de acordo com a ocupação exercida?

In [0]:
# Percentual de indivíduos com renda acima de 50K
# considerando simultaneamente escolaridade e ocupação.
#
# A análise permite verificar se a relação entre escolaridade
# e renda varia de acordo com a ocupação exercida.

from pyspark.sql.functions import count, sum, round

education_occupation_analysis = (
    gold_df
    .groupBy("education", "occupation")
    .agg(
        count("*").alias("total"),
        sum("salary_binary").alias("acima_50k"),
        round(
            (sum("salary_binary") / count("*")) * 100,
            2
        ).alias("percentual_acima_50k")
    )
    .orderBy("education", "occupation")
)

display(education_occupation_analysis)

In [0]:
# Seleciona algumas das principais ocupações para facilitar
# a comparação entre escolaridade e renda.
#
# Foram escolhidas ocupações com grande quantidade de registros,
# evitando categorias muito pequenas que poderiam distorcer a análise.

from pyspark.sql.functions import col

education_occupation_chart = (
    education_occupation_analysis
    .filter(
        col("occupation").isin(
            "Exec-managerial",
            "Prof-specialty",
            "Sales",
            "Craft-repair",
            "Adm-clerical"
        )
    )
    .select(
        "education",
        "occupation",
        "percentual_acima_50k"
    )
)

display(education_occupation_chart)

In [0]:
# Organiza os níveis de escolaridade em uma ordem lógica
# e mantém apenas as categorias mais representativas para o gráfico.

from pyspark.sql.functions import trim, when, col

education_order = (
    education_occupation_chart
    .withColumn(
        "education",
        trim(col("education"))
    )
    .filter(
        col("education").isin(
            "HS-grad",
            "Some-college",
            "Assoc-voc",
            "Assoc-acdm",
            "Bachelors",
            "Masters",
            "Prof-school",
            "Doctorate"
        )
    )
    .withColumn(
        "ordem_education",
        when(col("education") == "HS-grad", 1)
        .when(col("education") == "Some-college", 2)
        .when(col("education") == "Assoc-voc", 3)
        .when(col("education") == "Assoc-acdm", 4)
        .when(col("education") == "Bachelors", 5)
        .when(col("education") == "Masters", 6)
        .when(col("education") == "Prof-school", 7)
        .when(col("education") == "Doctorate", 8)
    )
    .orderBy("ordem_education")
    .select(
        "education",
        "occupation",
        "percentual_acima_50k"
    )
)

display(education_order)

In [0]:
# Mantém a ordem correta dos níveis de escolaridade
# para facilitar a visualização no gráfico.

education_order = (
    education_occupation_chart
    .withColumn(
        "education",
        trim(col("education"))
    )
    .filter(
        col("education").isin(
            "HS-grad",
            "Some-college",
            "Assoc-voc",
            "Assoc-acdm",
            "Bachelors",
            "Masters",
            "Prof-school",
            "Doctorate"
        )
    )
    .withColumn(
        "ordem_education",
        when(col("education") == "HS-grad", 1)
        .when(col("education") == "Some-college", 2)
        .when(col("education") == "Assoc-voc", 3)
        .when(col("education") == "Assoc-acdm", 4)
        .when(col("education") == "Bachelors", 5)
        .when(col("education") == "Masters", 6)
        .when(col("education") == "Prof-school", 7)
        .when(col("education") == "Doctorate", 8)
    )
    .orderBy("ordem_education")
)

display(education_order)

In [0]:
# Seleciona as principais ocupações e os níveis de escolaridade
# utilizados na análise visual.

from pyspark.sql.functions import col, trim, when

education_order = (
    education_occupation_analysis
    .withColumn(
        "education",
        trim(col("education"))
    )
    .filter(
        col("occupation").isin(
            "Exec-managerial",
            "Prof-specialty",
            "Sales",
            "Craft-repair",
            "Adm-clerical"
        )
    )
    .filter(
        col("education").isin(
            "HS-grad",
            "Some-college",
            "Assoc-voc",
            "Assoc-acdm",
            "Bachelors",
            "Masters",
            "Prof-school",
            "Doctorate"
        )
    )
    .withColumn(
        "ordem_education",
        when(col("education") == "HS-grad", 1)
        .when(col("education") == "Some-college", 2)
        .when(col("education") == "Assoc-voc", 3)
        .when(col("education") == "Assoc-acdm", 4)
        .when(col("education") == "Bachelors", 5)
        .when(col("education") == "Masters", 6)
        .when(col("education") == "Prof-school", 7)
        .when(col("education") == "Doctorate", 8)
    )
    .orderBy("ordem_education")
    .select(
        "education",
        "occupation",
        "percentual_acima_50k"
    )
)

display(education_order)

In [0]:
# Cria um heatmap para visualizar a proporção de indivíduos
# com renda acima de 50K por escolaridade e ocupação.

import matplotlib.pyplot as plt
import pandas as pd

# Converte a tabela Spark para Pandas
heatmap_df = education_occupation_pivot.toPandas()

# Define a escolaridade como índice
heatmap_df = heatmap_df.set_index("education")

# Remove a coluna auxiliar, caso ainda esteja presente
heatmap_df = heatmap_df.drop(
    columns=["ordem_education"],
    errors="ignore"
)

# Cria o gráfico
plt.figure(figsize=(12, 7))

plt.imshow(
    heatmap_df,
    aspect="auto"
)

# Define os nomes dos eixos
plt.xticks(
    range(len(heatmap_df.columns)),
    heatmap_df.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(heatmap_df.index)),
    heatmap_df.index
)

plt.xlabel("Ocupação")
plt.ylabel("Escolaridade")
plt.title(
    "Percentual de indivíduos com renda acima de 50K\n"
    "por escolaridade e ocupação"
)

# Adiciona os valores dentro das células
for i in range(len(heatmap_df.index)):
    for j in range(len(heatmap_df.columns)):
        value = heatmap_df.iloc[i, j]

        if pd.notna(value):
            plt.text(
                j,
                i,
                f"{value:.1f}%",
                ha="center",
                va="center"
            )

plt.colorbar(
    label="Percentual acima de 50K (%)"
)

plt.tight_layout()
plt.show()